# Meeting Minutes + Argument & Conflict Detector
This notebook:
1. Transcribes audio using open source Whisper
2. Performs speaker diarization using pyannote
3. Generates standard meeting minutes using Llama 3.2
4. Detects conflicts, opposing views, and generates a Meeting Tension Score

In [ ]:
# Install dependencies
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6
!pip install -q pyannote.audio==3.1.1
!pip install -q pydub

In [ ]:
# Imports

import os
import torch
import json
from IPython.display import Markdown, display
from google.colab import drive, userdata
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TextStreamer,
    BitsAndBytesConfig,
    pipeline
)
from pyannote.audio import Pipeline as DiarizationPipeline

In [ ]:
# Constants

LLAMA = "meta-llama/Llama-3.2-3B-Instruct"
WHISPER_MODEL = "openai/whisper-medium.en"
DIARIZATION_MODEL = "pyannote/speaker-diarization-3.1"

In [ ]:
# Mount Google Drive and set audio file path

drive.mount("/content/drive")
audio_filename = "/content/drive/MyDrive/llms/denver_extract.mp3"

In [ ]:
# Login to HuggingFace
# Note: You need to accept pyannote model terms at:
# https://huggingface.co/pyannote/speaker-diarization-3.1

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

# STEP 1: Transcribe Audio (Whisper)

In [ ]:
# Transcribe using open source Whisper with timestamps
# We need return_timestamps=True so we can align speakers later

whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model=WHISPER_MODEL,
    dtype=torch.float16,
    device="cuda",
    return_timestamps=True
)

result = whisper_pipe(audio_filename)
transcription = result["text"]
chunks = result["chunks"]  # List of {text, timestamp: [start, end]}

print("Transcription complete!")
print(f"Total chunks: {len(chunks)}")
print("\nRaw transcription:")
print(transcription)

# STEP 2: Speaker Diarization (pyannote)
This labels each audio segment with a speaker ID (SPEAKER_00, SPEAKER_01, etc.)

In [ ]:
# Run speaker diarization

diarization_pipeline = DiarizationPipeline.from_pretrained(
    DIARIZATION_MODEL,
    use_auth_token=hf_token
)
diarization_pipeline = diarization_pipeline.to(torch.device("cuda"))

diarization = diarization_pipeline(audio_filename)

# Extract speaker segments: list of (start, end, speaker)
speaker_segments = []
for turn, _, speaker in diarization.itertracks(yield_label=True):
    speaker_segments.append({
        "start": round(turn.start, 2),
        "end": round(turn.end, 2),
        "speaker": speaker
    })

print(f"Found {len(set(s['speaker'] for s in speaker_segments))} speakers")
print("First 5 segments:")
for s in speaker_segments[:5]:
    print(s)

In [ ]:
# Align Whisper chunks with diarization speaker labels
# For each Whisper chunk, find the speaker who was talking at that timestamp

def get_speaker_for_chunk(chunk_start, chunk_end, speaker_segments):
    """Find which speaker has most overlap with a given time window."""
    speaker_time = {}
    for seg in speaker_segments:
        overlap_start = max(chunk_start, seg["start"])
        overlap_end = min(chunk_end, seg["end"])
        if overlap_end > overlap_start:
            overlap = overlap_end - overlap_start
            speaker = seg["speaker"]
            speaker_time[speaker] = speaker_time.get(speaker, 0) + overlap
    if not speaker_time:
        return "UNKNOWN"
    return max(speaker_time, key=speaker_time.get)


def format_timestamp(seconds):
    """Convert seconds to MM:SS format."""
    minutes = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{minutes:02d}:{secs:02d}"


# Build speaker-tagged transcript
speaker_tagged_lines = []
current_speaker = None
current_text = []
current_start = None

for chunk in chunks:
    start, end = chunk["timestamp"]
    if start is None:
        start = 0.0
    if end is None:
        end = start + 2.0

    speaker = get_speaker_for_chunk(start, end, speaker_segments)

    if speaker != current_speaker:
        if current_speaker is not None and current_text:
            speaker_tagged_lines.append({
                "timestamp": format_timestamp(current_start),
                "speaker": current_speaker,
                "text": " ".join(current_text).strip()
            })
        current_speaker = speaker
        current_text = [chunk["text"]]
        current_start = start
    else:
        current_text.append(chunk["text"])

# Append last segment
if current_speaker and current_text:
    speaker_tagged_lines.append({
        "timestamp": format_timestamp(current_start),
        "speaker": current_speaker,
        "text": " ".join(current_text).strip()
    })

# Build readable speaker-tagged transcript string
speaker_tagged_transcript = ""
for line in speaker_tagged_lines:
    speaker_tagged_transcript += f"[{line['timestamp']}] {line['speaker']}: {line['text']}\n"

print("Speaker-tagged transcript (first 20 lines):")
print("\n".join(speaker_tagged_transcript.split("\n")[:20]))

# STEP 3: Load Llama Model

In [ ]:
# Load Llama with 4-bit quantization

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    LLAMA,
    device_map="auto",
    quantization_config=quant_config
)

print("Model loaded!")

In [ ]:
# Helper function to run inference with Llama

def run_llama(system_message, user_prompt, max_new_tokens=2000, stream=True):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt}
    ]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer) if stream else None
    outputs = model.generate(inputs, max_new_tokens=max_new_tokens, streamer=streamer)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the assistant's reply
    if "assistant" in response:
        response = response.split("assistant")[-1].strip()
    return response

# STEP 4: Generate Standard Meeting Minutes

In [ ]:
minutes_system = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

minutes_prompt = f"""
Below is a speaker-tagged transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- A summary with attendees (use speaker labels), location and date
- Key discussion points
- Takeaways
- Action items with owners (use speaker labels)

Transcript:
{speaker_tagged_transcript}
"""

print("Generating meeting minutes...\n")
meeting_minutes = run_llama(minutes_system, minutes_prompt, max_new_tokens=2000)
display(Markdown(meeting_minutes))

# STEP 5: Argument & Conflict Detection

In [ ]:
conflict_system = """
You are an expert meeting analyst specializing in detecting arguments, disagreements,
and conflicts in meeting transcripts. You analyze tone, opposing viewpoints, and resolution.
Always respond in valid JSON only, with no extra text or markdown.
"""

conflict_prompt = f"""
Analyze the following speaker-tagged meeting transcript for conflicts and disagreements.

For each conflict found, extract:
- conflict_id: integer starting from 1
- topic: short topic label
- speakers_involved: list of speaker IDs
- positions: dict mapping each speaker to their stated position
- intensity: one of ["mild disagreement", "moderate conflict", "heated argument"]
- resolution: one of ["resolved", "partially resolved", "deferred", "unresolved"]
- start_timestamp: approximate timestamp when conflict began
- summary: 2-3 sentence summary of what was argued

Also provide:
- tension_score: integer 0-100 representing overall meeting tension
- tension_explanation: one sentence explaining the score
- dominant_speaker: speaker ID who drove most of the discussion
- most_contested_topic: the topic with the most disagreement

Return ONLY a JSON object with keys: conflicts (list), tension_score, tension_explanation,
dominant_speaker, most_contested_topic.

Transcript:
{speaker_tagged_transcript}
"""

print("Detecting conflicts...\n")
conflict_raw = run_llama(conflict_system, conflict_prompt, max_new_tokens=2000, stream=False)

In [ ]:
# Parse the JSON response

import re

def extract_json(text):
    """Extract JSON from model output, handling any surrounding text."""
    # Try to find JSON block
    json_match = re.search(r'\{.*\}', text, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass
    # If that fails, try parsing the whole string
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"JSON parsing failed: {e}")
        print("Raw output:", text[:500])
        return None

conflict_data = extract_json(conflict_raw)

if conflict_data:
    print(f"Found {len(conflict_data.get('conflicts', []))} conflict(s)")
    print(f"Tension Score: {conflict_data.get('tension_score', 'N/A')}/100")
else:
    print("Could not parse conflict data. Raw output:")
    print(conflict_raw)

In [ ]:
# Format the conflict report as markdown

def format_conflict_report(data):
    if not data:
        return "*Could not generate conflict report.*"

    lines = []
    lines.append("---")
    lines.append("## Argument & Conflict Analysis")
    lines.append("")

    # Tension score with visual bar
    score = data.get("tension_score", 0)
    filled = int(score / 10)
    bar = "█" * filled + "░" * (10 - filled)
    lines.append(f"### Meeting Tension Score: {score}/100")
    lines.append(f"`{bar}`")
    lines.append(f"> {data.get('tension_explanation', '')}")
    lines.append("")

    lines.append(f"**Dominant Speaker:** {data.get('dominant_speaker', 'N/A')}  ")
    lines.append(f"**Most Contested Topic:** {data.get('most_contested_topic', 'N/A')}")
    lines.append("")

    conflicts = data.get("conflicts", [])
    if not conflicts:
        lines.append("*No significant conflicts detected.*")
        return "\n".join(lines)

    lines.append(f"### Conflicts Detected: {len(conflicts)}")
    lines.append("")

    intensity_emoji = {
        "mild disagreement": "🟡",
        "moderate conflict": "🟠",
        "heated argument": "🔴"
    }
    resolution_emoji = {
        "resolved": "✅",
        "partially resolved": "🔄",
        "deferred": "📅",
        "unresolved": "❌"
    }

    for c in conflicts:
        intensity = c.get("intensity", "")
        resolution = c.get("resolution", "")
        i_emoji = intensity_emoji.get(intensity, "")
        r_emoji = resolution_emoji.get(resolution, "")

        lines.append(f"#### Conflict #{c.get('conflict_id', '')} — {c.get('topic', 'Unknown Topic')}")
        lines.append(f"- **Timestamp:** {c.get('start_timestamp', 'N/A')}")
        lines.append(f"- **Speakers:** {', '.join(c.get('speakers_involved', []))}")
        lines.append(f"- **Intensity:** {i_emoji} {intensity.title()}")
        lines.append(f"- **Resolution:** {r_emoji} {resolution.title()}")
        lines.append("")
        lines.append("**Positions:**")
        for speaker, position in c.get("positions", {}).items():
            lines.append(f"- *{speaker}:* {position}")
        lines.append("")
        lines.append(f"**Summary:** {c.get('summary', '')}")
        lines.append("")
        lines.append("---")
        lines.append("")

    return "\n".join(lines)


conflict_report = format_conflict_report(conflict_data)
display(Markdown(conflict_report))

# STEP 6: Full Combined Output

In [ ]:
# Combine meeting minutes + conflict report into one final document

full_output = f"""{meeting_minutes}

{conflict_report}
"""

display(Markdown(full_output))

In [ ]:
# Optional: Save full output to Google Drive

output_path = "/content/drive/MyDrive/llms/meeting_minutes_with_conflicts.md"

with open(output_path, "w") as f:
    f.write(full_output)

print(f"Saved to {output_path}")